# IMPORTS

In [20]:
import pymysql # working with MySQL
import pymongo # working with MongoDB
from datetime import datetime # for timestamps in logs

# GLOBAL CONSTANTS

In [21]:
String_delimiter = "-" * 75  # Lines for clean output formatting
String_delimiter_double = "=" * 75

PAGE_SIZE = 10  # Number of films per page (pagination)

MYSQL_HOST = "MySQL_host"  # MySQL settings
MYSQL_USER = "MySQL_user"
MYSQL_PASSWORD = "MySQL_password"
MYSQL_DB = "MYSQL_DB"
MYSQL_PORT = MYSQL_PORT

try:
    mongo_client = pymongo.MongoClient("mongo_client_URI")
    mongo_db = mongo_client["mongo_db"]
    collection = mongo_db["collection"]
    print("MongoDB connected")
except Exception as e:
    print(f"MongoDB connection error: {e}")

MongoDB подключена


# Connecting to MySQL and sakila database

In [ ]:
# Creating a single connection used throughout the program

try:
    connection = pymysql.connect(
        host=MYSQL_HOST,
        user=MYSQL_USER,
        password=MYSQL_PASSWORD,
        database=MYSQL_DB,
        port=MYSQL_PORT
    )
    print("MySQL connected")
except pymysql.Error as e:
    print(f"MySQL connection error: {e}")

# Search functions (MySQL)

## Search by keyword

In [23]:
def choose_film(key_word: str, offset: int) -> tuple:
    """
    Searches for films by keyword in title.

    key_word is the search term,
    offset is the starting position for pagination.
    """
    with connection.cursor() as cursor:

        # Count total number of results
        cursor.execute(
            """
            SELECT COUNT(*) AS total
            FROM film f
            JOIN film_category fc ON f.film_id = fc.film_id
            JOIN category c ON fc.category_id = c.category_id
            WHERE f.title LIKE %s
            """,
            (f"%{key_word.upper()}%",)
        )

        # fetchone() returns one row; [0] extracts the value
        total = cursor.fetchone()[0]

        cursor.execute(
            """
            SELECT f.film_id,
                   f.title AS title,
                   f.release_year AS year,
                   f.rating AS rating,
                   c.name AS genre
            FROM film f
            JOIN film_category fc ON f.film_id = fc.film_id
            JOIN category c ON fc.category_id = c.category_id
            WHERE f.title LIKE %s
            ORDER BY f.title
            LIMIT 10 OFFSET %s
            """,
            (f"%{key_word.upper()}%", offset)
        )

        results = cursor.fetchall()  # fetch all 10 rows at once

    print(String_delimiter)
    print(f"{'№':<4} {'Title':<40} {'Year':<6} {'Rating':<8} {'Genre'}")
    print(String_delimiter)

    for i, row in enumerate(results, start=offset + 1):
        print(f"{i:<4} {row[1]:<40} {row[2]:<6} {row[3]:<8} {row[4]}")

    print(String_delimiter)

    return results, total

## Search by genre and years

In [24]:
def get_genres() -> list:
    """
    Returns a list of all genres with year ranges.
    """
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT c.category_id,
                   c.name,
                   MIN(f.release_year) AS year_min,
                   MAX(f.release_year) AS year_max
            FROM category c
            JOIN film_category fc ON c.category_id = fc.category_id
            JOIN film f ON fc.film_id = f.film_id
            GROUP BY c.category_id, c.name
            ORDER BY c.name
            """
        )

        # return all genres with year range
        return cursor.fetchall()


def search_by_genre(
    genre_id: int,
    year_from: int,
    year_to: int,
    offset: int
) -> tuple:
    """
    Searches films by genre and year range.
    """
    with connection.cursor() as cursor:

        cursor.execute(
            """
            SELECT COUNT(*)
            FROM film f
            JOIN film_category fc ON f.film_id = fc.film_id
            WHERE fc.category_id = %s
              AND f.release_year BETWEEN %s AND %s
            """,
            (genre_id, year_from, year_to)
        )

        # fetch single row result (tuple)
        total = cursor.fetchone()[0]

        cursor.execute(
            """
            SELECT f.film_id,
                   f.title,
                   f.release_year,
                   fc.category_id,
                   f.rating
            FROM film f
            JOIN film_category fc ON f.film_id = fc.film_id
            WHERE fc.category_id = %s
              AND f.release_year BETWEEN %s AND %s
            ORDER BY f.release_year, f.title
            LIMIT 10 OFFSET %s
            """,
            (genre_id, year_from, year_to, offset)
        )

        results = cursor.fetchall()

    print(String_delimiter)
    print(
        f"{'№':<4} {'Title':<40} {'Year':<6} "
        f"{'Genre ID':<10} {'Rating'}"
    )
    print(String_delimiter)

    for i, row in enumerate(results, start=offset + 1):
        print(f"{i:<4} {row[1]:<40} {row[2]:<6} {row[3]:<10} {row[4]}")

    print(String_delimiter)

    return results, total

## Search by film rating

In [25]:
def get_ratings() -> list:
    """
    Returns a list of all available ratings from the database.
    """
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT DISTINCT rating
            FROM film
            ORDER BY rating
            """
        )

        ratings = cursor.fetchall()  # fetch all ratings
        return [row[0] for row in ratings]  # return only rating values


def search_by_rating(rating: str, offset: int) -> tuple:
    """
    Searches films by a specific rating (G, PG, PG-13, R, NC-17).
    """
    with connection.cursor() as cursor:

        cursor.execute(
            """
            SELECT COUNT(*)
            FROM film
            WHERE rating = %s
            """,
            (rating,)
        )

        # fetchone() returns one row; [0] extracts the value
        total = cursor.fetchone()[0]

        cursor.execute(
            """
            SELECT film_id, title, release_year, rating
            FROM film
            WHERE rating = %s
            LIMIT 10 OFFSET %s
            """,
            (rating, offset)
        )

        results = cursor.fetchall()

    print(String_delimiter)
    print(f"{'№':<4} {'Title':<40} {'Year':<6} {'Rating'}")
    print(String_delimiter)

    # pagination numbering
    for i, row in enumerate(results, start=offset + 1):
        print(f"{i:<4} {row[1]:<40} {row[2]:<6} {row[3]}")

    print(String_delimiter)

    return results, total

# Output functions with pagination

In [26]:
def paginate(search_func: callable, args: tuple, total: int) -> None:
    """
    Universal pagination for any search scenario.

    search_func: function used for searching
    args: search arguments (without offset)
    total: total number of found records
    """
    page = 0  # current offset (0 = first page, 10 = second, etc.)
    first = True  # first page already printed outside paginate

    while True:

        # Load and display next/previous page
        # First page is skipped because it was already printed
        if not first:
            search_func(*args, page)

        first = False

        # Display counter
        shown = min(page + PAGE_SIZE, total)
        print(f"Shown {shown} of {total}")

        # Determine current position
        is_first_page = (page == 0)
        is_last_page = (page + PAGE_SIZE >= total)

        # Show available navigation options
        if is_first_page:
            print("* — next page | 0 — return to menu")
        elif is_last_page:
            print("This is the last page.")
            print("< — previous page | 0 — return to menu")
        else:
            print("* — next | < — previous | 0 — menu")

        par = input("Your choice: ").strip()

        if par == "0":
            break
        elif par == "*" and not is_last_page:
            page += PAGE_SIZE
        elif par == "<" and not is_first_page:
            page -= PAGE_SIZE
        else:
            print("Use *, < or 0")

# Logging search queries to MongoDB

In [27]:
def log_write(
    search_type: str,
    params: dict,
    results_count: int
) -> None:
    """
    Writes a single search log entry into MongoDB.

    search_type: type of search ('keyword', 'genre', 'rating')
    params: dictionary with search parameters
    results_count: number of found films
    timestamp: time when the log is created
    """
    collection.insert_one(
        {
            "search_type": search_type,
            "params": params,
            "results_count": results_count,
            "timestamp": datetime.now()
        }
    )

In [28]:
def get_log() -> None:
    """
    Prints all MongoDB log entries.
    """
    print(String_delimiter_double)
    print(f"{'ID':<10} {'Search params':<30} {'Results':<12} {'Time'}")
    print(String_delimiter_double)

    # limit number of records
    for doc in collection.find().limit(50):
        print(
            f"{str(doc['_id'])[-6:]:<10} "
            f"{str(doc['params']):<30} "
            f"{doc['results_count']:<12} "
            f"{doc['timestamp']}"
        )

In [29]:
def ask_top_n() -> int:
    """
    Asks the user how many records to display (TOP-N).
    """
    while True:
        try:
            n = int(input("Enter a number (e.g. 5): "))
            if n > 0:
                return n
            print("Number must be greater than 0.")
        except ValueError:
            print("Please enter a valid number!")

In [30]:
def stats_keyword(n: int) -> None:
    """
    Displays TOP-N most popular keyword searches.
    """
    pipeline = [
        # Keep only documents with search_type 'keyword'
        {"$match": {"search_type": "keyword"}},

        # Group by keyword, count occurrences, track last search time
        {
            "$group": {
                "_id": "$params.keyword",
                "total": {"$sum": 1},
                "last_time": {"$max": "$timestamp"}
            }
        },

        {"$sort": {"total": -1}},
        {"$limit": n}
    ]

    # Execute aggregation pipeline
    results = list(collection.aggregate(pipeline))

    print(String_delimiter_double)
    print(f"TOP-{n} MOST POPULAR KEYWORD SEARCHES:")
    print(String_delimiter_double)
    print(f"{'№':<4} {'Keyword':<25} {'Searches':<10} {'Last time'}")
    print(String_delimiter)

    for i, item in enumerate(results, start=1):
        print(
            f"{i:<4} {str(item['_id']):<25} "
            f"{item['total']:<10} "
            f"{item['last_time']}"
        )

    print(String_delimiter_double)

In [31]:
def stats_genre_years(n: int) -> None:
    """
    Displays TOP-N most popular genre-year searches.
    """
    pipeline = [
        # Keep only documents with search_type 'genre'
        {"$match": {"search_type": "genre"}},

        # Group by full params dictionary
        {
            "$group": {
                "_id": "$params",
                "total": {"$sum": 1},
                "last_time": {"$max": "$timestamp"}
            }
        },

        {"$sort": {"total": -1}},
        {"$limit": n}
    ]

    # Run aggregation pipeline
    results = list(collection.aggregate(pipeline))

    # Fetch genre names from MySQL
    genre_names = {}
    with connection.cursor() as cursor:
        cursor.execute("SELECT category_id, name FROM category")
        for row in cursor.fetchall():
            genre_names[row[0]] = row[1]

    print(String_delimiter_double)
    print(f"TOP-{n} MOST POPULAR GENRE-YEAR SEARCHES:")
    print(String_delimiter_double)

    print(
        f"{'№':<4} {'Genre':<20} {'ID':<6} {'Years':<15} "
        f"{'Searches':<10} {'Last time'}"
    )

    print(String_delimiter)

    for i, item in enumerate(results, start=1):
        p = item["_id"]
        genre_id = p.get("genre_id", "?")
        genre_name = genre_names.get(genre_id, "?")
        years = f"{p.get('year_from', '?')} - {p.get('year_to', '?')}"

        print(
            f"{i:<4} {genre_name:<20} {str(genre_id):<6} {years:<15} "
            f"{item['total']:<10} "
            f"{item['last_time']}"
        )

    print(String_delimiter_double)

In [32]:
def stats_recent(n: int) -> None:
    """
    Displays TOP-N most recent unique searches.
    """
    pipeline = [
        # Sort by timestamp (newest first)
        {"$sort": {"timestamp": -1}},

        # Group by params, keep only unique searches
        {
            "$group": {
                "_id": "$params",
                "search_type": {"$first": "$search_type"},
                "timestamp": {"$first": "$timestamp"}
            }
        },

        {"$sort": {"timestamp": -1}},
        {"$limit": n}
    ]

    results = list(collection.aggregate(pipeline))

    print(String_delimiter_double)
    print(f"TOP-{n} MOST RECENT UNIQUE SEARCHES:")
    print(String_delimiter_double)

    print(f"{'No':<4} {'Type':<12} {'Parameters':<30} {'Time'}")
    print(String_delimiter)

    for i, item in enumerate(results, start=1):
        print(
            f"{i:<4} {item['search_type']:<12} "
            f"{str(item['_id']):<30} "
            f"{item['timestamp']}"
        )

    print(String_delimiter_double)

In [33]:
def stats_popular(n: int) -> None:
    """
    Displays TOP-N most popular searches across all search types.
    """
    pipeline = [
        # Group by params and count occurrences
        {
            "$group": {
                "_id": "$params",
                "search_type": {"$first": "$search_type"},
                "total": {"$sum": 1},
                "last_time": {"$max": "$timestamp"}
            }
        },

        {"$sort": {"total": -1}},
        {"$limit": n}
    ]

    results = list(collection.aggregate(pipeline))

    print(String_delimiter_double)
    print(f"TOP-{n} MOST POPULAR SEARCHES:")
    print(String_delimiter_double)

    print(
        f"{'№':<4} {'Type':<12} {'Parameters':<30} "
        f"{'Searches':<10} {'Last time'}"
    )

    print(String_delimiter)

    for i, item in enumerate(results, start=1):
        print(
            f"{i:<4} {item['search_type']:<12} "
            f"{str(item['_id']):<30} "
            f"{item['total']:<10} "
            f"{item['last_time']}"
        )

    print(String_delimiter_double)

In [34]:
def mongoagr() -> None:
    """
    MongoDB statistics menu.

    Allows selection of TOP-N:
    - keyword searches
    - genre-year searches
    - most popular searches across all types
    - most recent unique searches
    """
    while True:
        print(String_delimiter_double)
        print(
            f"{String_delimiter[:22]}-SEARCH STATISTICS (MongoDB)-"
            f"{String_delimiter[:22]}"
        )
        print(String_delimiter_double)

        print("1 - TOP-N popular keyword searches")
        print("2 - TOP-N popular genre-year searches")
        print("3 - TOP-N most popular searches (all types)")
        print("4 - TOP-N most recent unique searches")
        print("5 - Exit")

        print(String_delimiter_double)

        # Input validation
        try:
            par = int(
                input("Select option for 'SEARCH STATISTICS' menu: ")
            )
            if par not in (1, 2, 3, 4, 5):
                raise ValueError
        except ValueError:
            print("Enter a number from 1 to 5!")
            continue

        if par == 5:
            break

        n = ask_top_n()  # number of records to display

        if par == 1:
            stats_keyword(n)
        elif par == 2:
            stats_genre_years(n)
        elif par == 3:
            stats_popular(n)
        elif par == 4:
            stats_recent(n)

# Scenario builders

In [35]:
def scenario_keyword() -> None:
    """
    Scenario 1: search by keyword.
    """
    print(String_delimiter_double)
    print("KEYWORD SEARCH")
    print(String_delimiter_double)

    # Get non-empty input
    key_word = ""
    while not key_word:
        key_word = input("Enter keyword: ").strip().upper()
        if not key_word:
            print("Empty input! Try again.")

    # Initial search to get total results
    _, total = choose_film(key_word, 0)

    if total == 0:
        print(f'No results found for "{key_word}".')
        return

    # Log search in MongoDB
    log_write("keyword", {"keyword": key_word}, total)

    # Start pagination
    paginate(choose_film, (key_word,), total)

In [36]:
def scenario_genre_years() -> None:
    """
    Scenario 2: search by genre and years.
    """
    print(String_delimiter_double)
    print("GENRE AND YEAR SEARCH")
    print(String_delimiter_double)

    # Show available genres
    genres = get_genres()
    print(f"{'ID':<6} {'Genre':<25} {'Years'}")

    for g in genres:
        print(f"{g[0]:<6} {g[1]:<25} {g[2]}–{g[3]}")

    # Validate genre ID
    valid_ids = [g[0] for g in genres]
    genre_id = None

    while genre_id not in valid_ids:
        try:
            genre_id = int(input("Enter genre ID: "))
            if genre_id not in valid_ids:
                print("Invalid genre ID.")
        except ValueError:
            print("Please enter a number!")

    # Validate year range
    year_from, year_to = 1, 0  # invalid initial state

    while year_from > year_to:
        try:
            year_from = int(input("From year: "))
            year_to = int(input("To year:   "))

            if year_from > year_to:
                print("Start year cannot be greater than end year.")

        except ValueError:
            print("Please enter valid numbers!")
            year_from, year_to = 1, 0

    # Initial query to get total results
    _, total = search_by_genre(genre_id, year_from, year_to, 0)

    if total == 0:
        print("No results found for these parameters.")
        return

    # Log query
    log_write(
        "genre",
        {"genre_id": genre_id, "year_from": year_from, "year_to": year_to},
        total
    )

    # Start pagination
    paginate(search_by_genre, (genre_id, year_from, year_to), total)

In [37]:
def scenario_rating() -> None:
    """
    Scenario 3: search by film rating.
    """
    print(String_delimiter_double)
    print("RATING SEARCH")
    print(String_delimiter_double)

    # Show available ratings
    ratings = get_ratings()
    print("Available ratings:", " | ".join(ratings))

    # Validate rating input
    rating = ""
    while rating not in ratings:
        rating = input("Enter rating: ").strip().upper()
        if rating not in ratings:
            print(f"Invalid rating. Choose from: {', '.join(ratings)}")

    # Initial search to get total results
    _, total = search_by_rating(rating, 0)

    if total == 0:
        print(f'No results found for rating "{rating}".')
        return

    # Log to MongoDB
    log_write("rating", {"rating": rating}, total)

    # Start pagination
    paginate(search_by_rating, (rating,), total)

# Main menu



In [ ]:
def main() -> None:
    while True:
        print(String_delimiter_double)
        print("1. Search by keyword")
        print("2. Search by genre and years")
        print("3. Search by rating")
        print("4. Statistics")
        print("5. Exit")
        print(String_delimiter_double)

        # Input validation (1–5 only)
        try:
            par = int(input("Select option: "))
            if par not in (1, 2, 3, 4, 5):
                raise ValueError
        except ValueError:
            print("Enter a number from 1 to 5!")
            continue

        if par == 1:
            scenario_keyword()
        elif par == 2:
            scenario_genre_years()
        elif par == 3:
            scenario_rating()
        elif par == 4:
            mongoagr()
        elif par == 5:
            print("Goodbye!")

            # Close MySQL connection safely
            if connection:
                connection.close()

            break


if __name__ == "__main__":
    main()

1. Поиск по ключевому слову
2. Поиск по жанру и годам
3. Поиск по рейтингу фильма
4. Статистика
5. Выход


Выбор:  2


ПОИСК ПО ЖАНРУ И ГОДАМ
ID     Жанр                      Годы
1      Action                    1990–2025
2      Animation                 1990–2025
3      Children                  1990–2025
4      Classics                  1990–2025
5      Comedy                    1990–2024
6      Documentary               1990–2025
7      Drama                     1990–2025
8      Family                    1990–2025
9      Foreign                   1990–2025
10     Games                     1991–2025
11     Horror                    1990–2024
12     Music                     1990–2025
13     New                       1990–2025
14     Sci-Fi                    1990–2025
15     Sports                    1990–2023
16     Travel                    1990–2025


Введи ID жанра:  e


Нужно ввести число!


...

Выбор:  4


-----------------------СТАТИСТИКА ЗАПРОСОВ (MongoDB)-----------------------
1 - TOP-N популярных запросов по {keyword}
2 - TOP-N популярных запросов по {genres-years}
3 - TOP-N самых популярных запросов (все типы)
4 - TOP-N последних уникальных запросов
5 - Выход


Выбери пункт меню для блока 'СТАТИСТИКА ЗАПРОСОВ':  4
Введи своё число (например: 5):  5


ТОП-5 ПОСЛЕДНИХ УНИКАЛЬНЫХ ЗАПРОСОВ:
No   Тип          Параметры                      Время
---------------------------------------------------------------------------
1    rating       {'rating': 'R'}                2026-03-13 15:52:17.327000
2    genre        {'genre_id': 3, 'year_from': 2016, 'year_to': 2017} 2026-03-13 15:51:43.706000
3    keyword      {'keyword': 'TITANIC'}         2026-03-13 15:50:10.283000
4    rating       {'rating': 'G'}                2026-03-13 10:52:51.533000
5    genre        {'genre_id': 1, 'year_from': 1990, 'year_to': 2023} 2026-03-13 10:52:12.238000
-----------------------СТАТИСТИКА ЗАПРОСОВ (MongoDB)-----------------------
1 - TOP-N популярных запросов по {keyword}
2 - TOP-N популярных запросов по {genres-years}
3 - TOP-N самых популярных запросов (все типы)
4 - TOP-N последних уникальных запросов
5 - Выход


Выбери пункт меню для блока 'СТАТИСТИКА ЗАПРОСОВ':   3
